# L_0.3.0 — 三實體 + 可插拔機制架構：AI 前沿天花板

> 類別:exp-notebook | 狀態:frozen | 版本:L_0.3.0(minor:新架構+新機制) | 日期:2026-07-05
> 研究動機:`docs/10_concept_v5_open-endgame.md` §2.2(F 天花板)、§2.3(labor_share)。
> **與上一版差異**:
> 1. 引擎從單檔 `model.py` 重構為「三實體(Human/AI 前沿 Matrix/Market)+ 可插拔機制」架構(`src/labor_sim/{config,metrics,entities/*,mechanisms/*,world}.py`)。
> 2. 新機制:AI 前沿天花板 `F_ceiling`(None=舊行為線性 F=F0+r·t；有限值=遞迴飽和 dF/dt=r·(1-F/ceiling))。
> 3. 新指標:`labor_share`(勞動所得份額，Engels' Pause 型體制的偵測用)、`near_frontier_share`(中介職業帶人口比例)。
> 4. `classify_regime` 新增第四種終局 `human_premium`(前沿停滯下的人類溢價)。
> **驗收**:三情境×五指標與 `results/L_0.2.2/summary.json` 之 `baseline_v022` 一致(容差 1e-9)；F_ceiling=None 時退化為舊行為逐位一致。

### 1. 設定
目的:版本字串(顯式 L_0.3.0)、輸出路徑(沿用 `labor_sim.output`)、CJK 字型、matplotlib Agg 後端、匯入新引擎套件。
驗證:cell 末印出 VERSION 與 ROOT；`results/L_0.3.0/` 目錄建立成功。

In [1]:
from __future__ import annotations
import os, sys, json, time

# 顯式指定本 notebook 的輸出版本(優先於 output.py 的模組預設 L_0.2.0)
os.environ["LSIM_VERSION"] = "L_0.3.0"
VERSION = "L_0.3.0"

ROOT = (os.path.abspath(os.path.join(os.getcwd(), ".."))
        if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import matplotlib
matplotlib.use("Agg")                     # headless，凍結三閘門 9.5
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

_avail = {f.name for f in font_manager.fontManager.ttflist}
for _name in ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC", "SimHei"]:
    if _name in _avail:
        matplotlib.rcParams["font.sans-serif"] = [_name]
        break
matplotlib.rcParams["axes.unicode_minus"] = False

from labor_sim.config import Config
from labor_sim.world import World, WorldResult, run_sim, classify_regime
from labor_sim.entities.market import Market
from labor_sim import output as out

RESULTS = out.results_dir(VERSION)
print(f"VERSION={VERSION}  ROOT={ROOT}  RESULTS={RESULTS}")

VERSION=L_0.3.0  ROOT=D:\Project\sideProject\labor_market_simulation  RESULTS=D:\Project\sideProject\labor_market_simulation\results\L_0.3.0


### 2. 引擎驗證(不變量，凍結閘門 9.4-1)
目的:用 assert 固定新引擎的基本性質——錯一條就不准往下跑(raise，不只是 warn)。
驗證項:
1. 就業率值域 [0,1]、收入非負。
2. `F_ceiling=None` 時 F 嚴格遞增，且與閉式解 `F0+r*t` 逐位相等(不是「近似」)。
3. `F_ceiling` 有限時 F 漸近、永不超過 ceiling。
4. 飽和學習(`ability_ceiling` 有限)不破天花板，即使 `F_ceiling` 同時開啟。
5. 同 seed 完全重現(逐位相等，跑兩次比較)。
Pseudocode:
```
组 A(ceiling=None): 斷言 F == F0 + r*arange(steps) 逐位相等
组 B(ceiling=有限):  斷言 F 單調不減 且 max(F) <= ceiling + 1e-9
组 C: 兩次同參數同 seed 跑，斷言 employment_rate 陣列 array_equal
```

In [2]:
_p_base = Config(r=0.004, human_learning=0.004, tasks_per_worker=1.0, ability_ceiling=3.0, seed=0)

# --- 組 A: F_ceiling=None 退化為閉式解，逐位相等 ---
_res_a = run_sim(_p_base)
_t = np.arange(_p_base.steps)
_F_closed = _p_base.F0 + _p_base.r * _t
assert np.array_equal(_res_a.F, _F_closed), "F_ceiling=None 時必須與閉式解逐位相等"
assert np.all(np.diff(_res_a.F) > 0), "F 必須嚴格遞增"
assert np.all((_res_a.employment_rate >= 0) & (_res_a.employment_rate <= 1))
assert _res_a.history_earnings_last.min() >= 0
assert _res_a.final_ability.max() <= 3.0 + 1e-9, "飽和學習不可破 ability_ceiling"

# --- 組 B: F_ceiling 有限，漸近且不超過 ceiling ---
_p_ceil = Config(r=0.004, human_learning=0.004, tasks_per_worker=1.0,
                 ability_ceiling=3.0, F_ceiling=0.5, seed=0)
_res_b = run_sim(_p_ceil)
assert _res_b.F.max() <= 0.5 + 1e-9, "F 不可超過 ceiling"
assert np.all(np.diff(_res_b.F) >= -1e-12), "F 應單調不減(遞迴飽和)"
assert _res_b.final_ability.max() <= 3.0 + 1e-9, "ceiling 併存下，飽和學習仍不可破 ability_ceiling"

# --- 組 C: 同 seed 逐位重現 ---
_res_c = run_sim(Config(r=0.004, human_learning=0.004, tasks_per_worker=1.0,
                         ability_ceiling=3.0, F_ceiling=0.5, seed=0))
assert np.array_equal(_res_b.employment_rate, _res_c.employment_rate), "同 seed 必須逐位重現"
assert np.array_equal(_res_b.F, _res_c.F)

print("引擎不變量:全部通過(F 閉式解逐位一致、ceiling 漸近不逾界、同 seed 重現)")

引擎不變量:全部通過(F 閉式解逐位一致、ceiling 漸近不逾界、同 seed 重現)


### 3. 回歸驗證(凍結閘門 9.4-2，硬阻擋)：重現 L_0.2.2 baseline
目的:證明重構**零漂移**——三情境(崩潰/拉鋸/重配置)×五項指標，在 `F_ceiling=None` 下必須與
`results/L_0.2.2/summary.json` 的 `baseline_v022` 一致(相對容差 1e-9)；regime 亦須一致。
驗證:任何一項不合立即 AssertionError，不靜默吞掉。

In [3]:
CALIB = dict(tasks_per_worker=1.0, ability_ceiling=3.0, F_ceiling=None)
SCENARIOS = {
    "崩潰 r>>g (無學習)": dict(r=0.005, human_learning=0.000),
    "拉鋸 r~=g":          dict(r=0.004, human_learning=0.004),
    "重配置 g>=r":        dict(r=0.003, human_learning=0.006),
}
METRICS = ["employment_t0", "employment_final", "gini_all_final",
           "gini_employed_final", "top10_final"]

with open(os.path.join(ROOT, "results", "L_0.2.2", "summary.json"), encoding="utf-8") as f:
    ref = json.load(f)["baseline_v022"]

# baseline_v022 的情境名用 unicode 符號(>>、~=)，本 notebook 用 ASCII 名 -> 對照表
NAME_MAP = {
    "崩潰 r>>g (無學習)": "崩潰 r≫g (無學習)",
    "拉鋸 r~=g":          "拉鋸 r≈g",
    "重配置 g>=r":        "重配置 g≥r",
}

results = {}
regression_report = {}
for name, kw in SCENARIOS.items():
    res = run_sim(Config(**kw, **CALIB))
    results[name] = res
    got = {
        "employment_t0": float(res.employment_rate[0]),
        "employment_final": float(res.employment_rate[-1]),
        "gini_all_final": float(res.gini[-1]),
        "gini_employed_final": float(res.gini_employed[-1]),
        "top10_final": float(res.top10_share[-1]),
    }
    exp = ref[NAME_MAP[name]]
    assert res.regime == exp["regime"], (name, res.regime, exp["regime"])
    max_rel_drift = 0.0
    for m in METRICS:
        tol = 1e-9 * max(1.0, abs(exp[m]))
        d = abs(got[m] - exp[m])
        max_rel_drift = max(max_rel_drift, d / max(1.0, abs(exp[m])))
        assert d <= tol, (name, m, got[m], exp[m], d)
    regression_report[name] = {"regime": res.regime, **got, "max_rel_drift": max_rel_drift}
    print(f"{name:22s} regime={res.regime:13s} emp_end={got['employment_final']:.4f}  "
          f"max_rel_drift={max_rel_drift:.3e}  MATCH")

print("回歸閘門:通過(重構零漂移，三情境x五指標全部與 L_0.2.2 一致，容差 1e-9)")

崩潰 r>>g (無學習)          regime=collapse      emp_end=0.0210  max_rel_drift=0.000e+00  MATCH


拉鋸 r~=g                regime=reallocation  emp_end=0.6985  max_rel_drift=0.000e+00  MATCH


重配置 g>=r               regime=reallocation  emp_end=0.9730  max_rel_drift=0.000e+00  MATCH
回歸閘門:通過(重構零漂移，三情境x五指標全部與 L_0.2.2 一致，容差 1e-9)


### 4. 實驗一：(r, F_ceiling) 相圖
目的:掃 (r, F_ceiling) 平面，g 固定 0.004，其餘同回歸情境校準；記錄末段 regime，
檢查是否出現第三(第四)種終局 `human_premium`(可證偽假設 H1)。
驗證:輸出離散色階熱圖 + 對應 CSV(`save_grid`)；summary 記錄出現的 regime 集合。
密度:9(r) x 9(F_ceiling)，F_ceiling 軸含 None(以 inf 代表，繪圖另標)。

In [4]:
R_GRID = np.linspace(0.002, 0.006, 9)
CEIL_FINITE = np.linspace(0.4, 1.2, 8)   # 8 個有限值 + 1 個 None(inf) = 9
CEIL_GRID_LABELS = list(CEIL_FINITE) + [np.inf]   # inf 代表 F_ceiling=None

REGIME_CODE = {"collapse": 0, "reallocation": 1, "concentration": 2, "human_premium": 3}
REGIME_NAMES = ["collapse", "reallocation", "concentration", "human_premium"]
REGIME_COLORS = ["#8B0000", "#4C72B0", "#DAA520", "#2E8B57"]

CALIB_G = dict(tasks_per_worker=1.0, ability_ceiling=3.0, human_learning=0.004)

Z1 = np.zeros((len(CEIL_GRID_LABELS), len(R_GRID)))
seen_regimes_1 = set()
_t0 = time.time()
for i, ceil in enumerate(CEIL_GRID_LABELS):
    fc = None if np.isinf(ceil) else float(ceil)
    for j, r in enumerate(R_GRID):
        res = run_sim(Config(r=float(r), F_ceiling=fc, **CALIB_G))
        Z1[i, j] = REGIME_CODE[res.regime]
        seen_regimes_1.add(res.regime)
_elapsed1 = time.time() - _t0
print(f"實驗一掃描完成:{len(R_GRID)}x{len(CEIL_GRID_LABELS)}={len(R_GRID)*len(CEIL_GRID_LABELS)} 格，耗時 {_elapsed1:.1f}s")
print("出現的 regime:", sorted(seen_regimes_1))

實驗一掃描完成:9x9=81 格，耗時 45.5s
出現的 regime: ['collapse', 'human_premium', 'reallocation']


In [5]:
# 圖:離散色階熱圖(F_ceiling 軸用 index，inf 顯示為 'None')
fig, ax = plt.subplots(figsize=(8, 6))
cmap = ListedColormap(REGIME_COLORS)
norm = BoundaryNorm(np.arange(-0.5, 4.5, 1), cmap.N)
im = ax.imshow(Z1, origin="lower", aspect="auto", cmap=cmap, norm=norm,
               extent=[R_GRID[0], R_GRID[-1], -0.5, len(CEIL_GRID_LABELS) - 0.5])
ax.set_yticks(range(len(CEIL_GRID_LABELS)))
ax.set_yticklabels([f"{c:.2f}" if np.isfinite(c) else "None(inf)" for c in CEIL_GRID_LABELS])
ax.set_xlabel("r (AI frontier speed)")
ax.set_ylabel("F_ceiling")
ax.set_title("Exp1: (r, F_ceiling) phase diagram, g=0.004 fixed")
legend_items = [Patch(facecolor=REGIME_COLORS[k], label=REGIME_NAMES[k]) for k in sorted(REGIME_CODE.values())]
ax.legend(handles=legend_items, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=9)
fig.tight_layout()
fig.savefig(out.fig_path("phase_r_ceiling.png", VERSION), dpi=120)
plt.show()

# CSV(long-form via save_grid；y 軸用有限代理值，inf 另存一欄標記)
y_numeric = np.array([c if np.isfinite(c) else -1.0 for c in CEIL_GRID_LABELS])  # -1 代表 None
out.save_grid("phase_r_ceiling.csv", x=R_GRID, y=y_numeric, Z=Z1,
              xlabel="r", ylabel="F_ceiling(-1=None)", value="regime_code", v=VERSION)
print("已存圖與 CSV")

已存圖與 CSV


C:\Users\user\AppData\Local\Temp\ipykernel_51280\3109849002.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. 實驗二：(g, F_ceiling) 相圖
目的:同法，r 固定 0.004，掃 human_learning(g) 與 F_ceiling。
驗證:輸出熱圖 + CSV；summary 記錄出現的 regime 集合。
密度:9(g) x 9(F_ceiling)。

In [6]:
G_GRID = np.linspace(0.0, 0.008, 9)
CALIB_R = dict(tasks_per_worker=1.0, ability_ceiling=3.0, r=0.004)

Z2 = np.zeros((len(CEIL_GRID_LABELS), len(G_GRID)))
seen_regimes_2 = set()
_t0 = time.time()
for i, ceil in enumerate(CEIL_GRID_LABELS):
    fc = None if np.isinf(ceil) else float(ceil)
    for j, g in enumerate(G_GRID):
        res = run_sim(Config(human_learning=float(g), F_ceiling=fc, **CALIB_R))
        Z2[i, j] = REGIME_CODE[res.regime]
        seen_regimes_2.add(res.regime)
_elapsed2 = time.time() - _t0
print(f"實驗二掃描完成:{len(G_GRID)}x{len(CEIL_GRID_LABELS)}={len(G_GRID)*len(CEIL_GRID_LABELS)} 格，耗時 {_elapsed2:.1f}s")
print("出現的 regime:", sorted(seen_regimes_2))

實驗二掃描完成:9x9=81 格，耗時 53.2s
出現的 regime: ['collapse', 'human_premium', 'reallocation']


In [7]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(Z2, origin="lower", aspect="auto", cmap=cmap, norm=norm,
               extent=[G_GRID[0], G_GRID[-1], -0.5, len(CEIL_GRID_LABELS) - 0.5])
ax.set_yticks(range(len(CEIL_GRID_LABELS)))
ax.set_yticklabels([f"{c:.2f}" if np.isfinite(c) else "None(inf)" for c in CEIL_GRID_LABELS])
ax.set_xlabel("g (human_learning)")
ax.set_ylabel("F_ceiling")
ax.set_title("Exp2: (g, F_ceiling) phase diagram, r=0.004 fixed")
ax.legend(handles=legend_items, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=9)
fig.tight_layout()
fig.savefig(out.fig_path("phase_g_ceiling.png", VERSION), dpi=120)
plt.show()

y_numeric2 = np.array([c if np.isfinite(c) else -1.0 for c in CEIL_GRID_LABELS])
out.save_grid("phase_g_ceiling.csv", x=G_GRID, y=y_numeric2, Z=Z2,
              xlabel="g", ylabel="F_ceiling(-1=None)", value="regime_code", v=VERSION)
print("已存圖與 CSV")

已存圖與 CSV


C:\Users\user\AppData\Local\Temp\ipykernel_51280\2812219593.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6. 實驗三：Engels' Pause 探針
目的:在代表性有限天花板參數下，畫 employment_rate 與 labor_share 雙時間序列(單一情境)，
標註「就業穩(末段>=0.7)但 labor_share 從峰值下降」的幅度(可證偽假設 H2)。
驗證:輸出雙序列圖 + CSV；summary 記錄峰值、末值、降幅(百分點)。

In [8]:
ENGELS_PARAMS = dict(r=0.004, human_learning=0.004, tasks_per_worker=1.0,
                     ability_ceiling=3.0, F_ceiling=0.5, seed=0)
res_engels = run_sim(Config(**ENGELS_PARAMS))

emp = res_engels.employment_rate
ls = res_engels.labor_share
tail_emp = emp[-12:].mean()
ls_peak = ls.max()
ls_final = ls[-1]
ls_drop_pp = (ls_peak - ls_final) * 100.0

fig, ax1 = plt.subplots(figsize=(9, 5))
months = np.arange(emp.size)
ax1.plot(months, emp, color="#4C72B0", lw=2, label="employment_rate")
ax1.set_xlabel("month")
ax1.set_ylabel("employment_rate", color="#4C72B0")
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.plot(months, ls, color="#C44E52", lw=2, label="labor_share")
ax2.set_ylabel("labor_share", color="#C44E52")
ax1.set_title(f"Exp3: Engels' Pause probe (F_ceiling={ENGELS_PARAMS['F_ceiling']}) | "
              f"tail_emp={tail_emp:.3f}, labor_share drop={ls_drop_pp:.1f}pp")
fig.tight_layout()
fig.savefig(out.fig_path("engels_pause_probe.png", VERSION), dpi=120)
plt.show()

out.save_columns("engels_pause_probe.csv",
                 {"month": months, "employment_rate": emp, "labor_share": ls}, v=VERSION)

print(f"末段就業率(近12月均值)={tail_emp:.4f}  labor_share 峰值={ls_peak:.4f}  "
      f"末值={ls_final:.4f}  降幅={ls_drop_pp:.2f} 個百分點")

末段就業率(近12月均值)=0.9715  labor_share 峰值=0.9807  末值=0.8405  降幅=14.01 個百分點


C:\Users\user\AppData\Local\Temp\ipykernel_51280\1139635465.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7. 實驗四：中介職業弧線
目的:近前沿人口份額(配對到 σ∈[F, F+0.2] 任務的工人比例，`near_frontier_share`，由主引擎逐步記錄)
隨時間變化，檢查是否「先升後降」(校準點:中介職業弧線，`docs/10_concept_v5_open-endgame.md` 表格)。
驗證:輸出時間序列 CSV；summary 記錄峰值位置與是否先升後降。

In [9]:
ARC_PARAMS = dict(r=0.004, human_learning=0.004, tasks_per_worker=1.0,
                  ability_ceiling=3.0, F_ceiling=0.5, seed=0)
res_arc = run_sim(Config(**ARC_PARAMS))
near_frontier_hist = res_arc.near_frontier_share

peak_idx = int(np.argmax(near_frontier_hist))
rises_then_falls = bool(0 < peak_idx < near_frontier_hist.size - 1
                        and near_frontier_hist[peak_idx] > near_frontier_hist[0]
                        and near_frontier_hist[peak_idx] > near_frontier_hist[-1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(near_frontier_hist.size), near_frontier_hist, color="#55A868", lw=2)
ax.axvline(peak_idx, color="grey", ls="--", lw=1)
ax.set_xlabel("month")
ax.set_ylabel("near-frontier population share")
ax.set_title(f"Exp4: intermediate occupation arc (peak at month {peak_idx})")
fig.tight_layout()
fig.savefig(out.fig_path("intermediate_arc.png", VERSION), dpi=120)
plt.show()

out.save_columns("intermediate_arc.csv",
                 {"month": np.arange(near_frontier_hist.size), "near_frontier_share": near_frontier_hist}, v=VERSION)
print(f"峰值位置=month {peak_idx} (值={near_frontier_hist[peak_idx]:.4f})  先升後降={rises_then_falls}")

峰值位置=month 10 (值=0.2590)  先升後降=True


C:\Users\user\AppData\Local\Temp\ipykernel_51280\453384237.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8. 敏感度：ability_ceiling ∈ {2.0, 3.0, 5.0}
目的:三個回歸情境在不同 ability_ceiling 下，regime 是否翻轉。
驗證:輸出表格式結果；summary 記錄是否有任何翻轉。

In [10]:
SENS_CEILINGS = [2.0, 3.0, 5.0]
sensitivity_report = {}
any_flip = False
for name, kw in SCENARIOS.items():
    regimes = {}
    for ac in SENS_CEILINGS:
        res = run_sim(Config(**kw, tasks_per_worker=1.0, ability_ceiling=ac, F_ceiling=None))
        regimes[ac] = res.regime
    flipped = len(set(regimes.values())) > 1
    any_flip = any_flip or flipped
    sensitivity_report[name] = {"regimes_by_ability_ceiling": regimes, "flipped": flipped}
    print(f"{name:22s} " + "  ".join(f"ac={ac}:{r}" for ac, r in regimes.items()) + f"  flipped={flipped}")

print(f"\n任一情境翻轉regime: {any_flip}")

崩潰 r>>g (無學習)          ac=2.0:collapse  ac=3.0:collapse  ac=5.0:collapse  flipped=False


拉鋸 r~=g                ac=2.0:reallocation  ac=3.0:reallocation  ac=5.0:reallocation  flipped=False


重配置 g>=r               ac=2.0:reallocation  ac=3.0:reallocation  ac=5.0:reallocation  flipped=False

任一情境翻轉regime: False


### 9. 圖與輸出彙總
目的:把四個實驗與回歸驗證的純量結論寫入 `summary.json`；記錄 `manifest.json`(參數/種子/git hash)。
驗證:`results/L_0.3.0/summary.json` 含各實驗 section；`manifest.json` 含本 notebook 執行紀錄。

In [11]:
out.update_summary("regression_vs_L_0.2.2", {
    "passed": True, "tolerance": "rel 1e-9", "metrics": METRICS,
    "scenarios": regression_report,
}, v=VERSION)

out.update_summary("exp1_phase_r_ceiling", {
    "grid_density": f"{len(R_GRID)}x{len(CEIL_GRID_LABELS)}",
    "r_range": [float(R_GRID[0]), float(R_GRID[-1])],
    "ceiling_values_finite": [float(c) for c in CEIL_FINITE],
    "ceiling_includes_none": True,
    "g_fixed": 0.004,
    "regimes_observed": sorted(seen_regimes_1),
    "human_premium_appeared": "human_premium" in seen_regimes_1,
    "elapsed_sec": _elapsed1,
}, v=VERSION)

out.update_summary("exp2_phase_g_ceiling", {
    "grid_density": f"{len(G_GRID)}x{len(CEIL_GRID_LABELS)}",
    "g_range": [float(G_GRID[0]), float(G_GRID[-1])],
    "ceiling_values_finite": [float(c) for c in CEIL_FINITE],
    "ceiling_includes_none": True,
    "r_fixed": 0.004,
    "regimes_observed": sorted(seen_regimes_2),
    "human_premium_appeared": "human_premium" in seen_regimes_2,
    "elapsed_sec": _elapsed2,
}, v=VERSION)

out.update_summary("exp3_engels_pause", {
    "params": ENGELS_PARAMS,
    "tail_employment_rate": float(tail_emp),
    "labor_share_peak": float(ls_peak),
    "labor_share_final": float(ls_final),
    "labor_share_drop_pp": float(ls_drop_pp),
    "employment_stable_ge_0.7": bool(tail_emp >= 0.7),
}, v=VERSION)

out.update_summary("exp4_intermediate_arc", {
    "params": ARC_PARAMS,
    "peak_month": peak_idx,
    "peak_value": float(near_frontier_hist[peak_idx]),
    "rises_then_falls": rises_then_falls,
}, v=VERSION)

out.update_summary("sensitivity_ability_ceiling", {
    "ceilings_tested": SENS_CEILINGS,
    "any_regime_flip": any_flip,
    "detail": sensitivity_report,
}, v=VERSION)

out.record_manifest(
    "L_0.3.0.ipynb",
    params={
        "CALIB_regression": CALIB,
        "SCENARIOS": SCENARIOS,
        "exp1": {"R_GRID": R_GRID.tolist(), "ceiling_finite": CEIL_FINITE.tolist(), "g_fixed": 0.004},
        "exp2": {"G_GRID": G_GRID.tolist(), "ceiling_finite": CEIL_FINITE.tolist(), "r_fixed": 0.004},
        "exp3": ENGELS_PARAMS,
        "exp4": ARC_PARAMS,
        "sensitivity": {"ability_ceilings": SENS_CEILINGS},
    },
    seeds=[0],
    notes="L_0.3.0:三實體+可插拔機制架構重構；新機制 F_ceiling；回歸零漂移對象=results/L_0.2.2/summary.json",
    v=VERSION,
)
print("產物已寫入", RESULTS)
print("git_hash:", out.git_hash())

產物已寫入 D:\Project\sideProject\labor_market_simulation\results\L_0.3.0
git_hash: dc75e2c


### 10. Headline
目的:本版 <=5 條含數字的結論(資料事實，不下研究論斷)，餵 `30_exp_ledger`。

In [12]:
print("1. 重構零漂移:三情境(collapse/reallocation/reallocation)x五指標與 L_0.2.2 一致(容差 1e-9)。")
print(f"2. 實驗一 (r, F_ceiling) 相圖 {len(R_GRID)}x{len(CEIL_GRID_LABELS)} 格出現 regime: {sorted(seen_regimes_1)}；"
      f"human_premium 出現={'human_premium' in seen_regimes_1}。")
print(f"3. 實驗二 (g, F_ceiling) 相圖 {len(G_GRID)}x{len(CEIL_GRID_LABELS)} 格出現 regime: {sorted(seen_regimes_2)}；"
      f"human_premium 出現={'human_premium' in seen_regimes_2}。")
print(f"4. Engels' Pause 探針:末段就業率={tail_emp:.3f}，labor_share 從峰值 {ls_peak:.3f} "
      f"降至 {ls_final:.3f}(降 {ls_drop_pp:.1f} 個百分點)。")
print(f"5. 中介職業弧線峰值於 month {peak_idx}，先升後降={rises_then_falls}；"
      f"敏感度(ability_ceiling)regime 翻轉={any_flip}。")

1. 重構零漂移:三情境(collapse/reallocation/reallocation)x五指標與 L_0.2.2 一致(容差 1e-9)。
2. 實驗一 (r, F_ceiling) 相圖 9x9 格出現 regime: ['collapse', 'human_premium', 'reallocation']；human_premium 出現=True。
3. 實驗二 (g, F_ceiling) 相圖 9x9 格出現 regime: ['collapse', 'human_premium', 'reallocation']；human_premium 出現=True。
4. Engels' Pause 探針:末段就業率=0.972，labor_share 從峰值 0.981 降至 0.841(降 14.0 個百分點)。
5. 中介職業弧線峰值於 month 10，先升後降=True；敏感度(ability_ceiling)regime 翻轉=False。
